# Level 1 · Step 7 — Teacher writes the ideal replies (Colab Pro L4)

**Goal:** for every training item built locally by `src/data/07_build_items.py` (business memory chunks +
hand-over message + last chat turns + a real customer question), the open teacher model writes the **ideal
WhatsApp reply**: only facts from the memory, the customer's language, short, or a hand-over when the memory
does not have the answer. Step 8 (`src/data/08_assemble_split_push.py`, local) then fact-checks every reply
and builds the train / validation / test files.

## What this notebook is — and is not

- **IS:** generation with an open teacher (Qwen2.5 Instruct, Apache-2.0), so the replies are ours to train on.
  The original AI replies in the chats came from paid third-party models and are **not** used as targets.
- **IS:** resumable. Every shard (1,000 items) is pushed to the private dataset repo
  `ankit-iiitdmj/mbg-answers-<export>` before the next one starts.
- **IS NOT:** fine-tuning. That is Level 3, a separate notebook.

## GPU

Pick **L4** (Runtime → Change runtime type). T4 → 7B teacher · **L4 → 14B** · A100 → 32B.
Expected time on the L4: **about 1–1.5 hours** for ~18,000 items. Disconnect the runtime when Cell 11 is done.

## First-shard health check

After the FIRST shard the run aborts if more than 5% of replies are empty, replies are far too long or short
on average, or (almost) everything / nothing is a hand-over — so a broken prompt never burns compute units.

In [ ]:
# Cell 2 — setup: clone repo + install dependencies (defensive)
REPO_URL  = "https://github.com/ankit8453/chat-AI-model.git"   # private repo -> asks for a GitHub token once
REPO_PATH = "/content/chat-AI-model"

import os, subprocess, sys, getpass

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", url, REPO_PATH], capture_output=True, text=True)

if os.path.isdir(REPO_PATH) and not os.path.isfile(os.path.join(REPO_PATH, "README.md")):
    print(f"Removing partial clone at {REPO_PATH} ...")
    subprocess.run(["rm", "-rf", REPO_PATH], check=True)
if not os.path.isdir(REPO_PATH):
    r = _clone(REPO_URL)
    if r.returncode != 0:
        print("Public clone failed (private repo?). Paste a GitHub token with read access:")
        tok = getpass.getpass("GitHub token: ")
        r = _clone(REPO_URL.replace("https://", f"https://{tok}@"))
        if r.returncode != 0:
            raise SystemExit("git clone failed:\n" + r.stderr[-1500:])
        subprocess.run(["git", "-C", REPO_PATH, "remote", "set-url", "origin", REPO_URL], check=False)
else:
    # Already cloned in this runtime -> pull the latest code (fixes pushed since the last run).
    r = subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only"], capture_output=True, text=True)
    if r.returncode != 0:
        tok = getpass.getpass("GitHub token (to pull updates): ")
        r = subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only",
                            REPO_URL.replace("https://", f"https://{tok}@"), "main"], capture_output=True, text=True)
    print("git pull:", (r.stdout or r.stderr).strip()[-300:])

print("code version:", subprocess.run(["git", "-C", REPO_PATH, "log", "-1", "--oneline"],
                                      capture_output=True, text=True).stdout.strip())
os.chdir(REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, "src"))
print("Repo root contents:", sorted(os.listdir(".")))

_pip_packages = ["huggingface_hub>=0.24", "transformers>=4.45", "accelerate>=0.33", "bitsandbytes>=0.43"]
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_pip_packages], capture_output=True, text=True)
if proc.returncode != 0:
    print("PIP STDERR (tail):\n" + proc.stderr[-3000:])
    raise SystemExit("pip install failed — see tail above.")

# vLLM is optional: much faster, but newer releases may not support the T4. If it fails we fall back.
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
print("vllm install:", "ok" if proc.returncode == 0 else "FAILED (will use transformers 4-bit fallback)")
print("setup ok")

In [ ]:
# Cell 3 — HF Hub login
from huggingface_hub import login, HfApi
login()  # Colab inline widget — paste your Write token

_whoami = HfApi().whoami()
assert _whoami["name"] == "ankit-iiitdmj", (
    f"HF Hub token belongs to {_whoami['name']!r}, expected 'ankit-iiitdmj'."
)
HF_NS = _whoami["name"]
print(f"HF Hub ok: user={HF_NS}")

In [ ]:
# Cell 4 — GPU check + teacher config
import subprocess
subprocess.run(["nvidia-smi"], check=False)
print()

from judge.engine import answer_config
cfg = answer_config()
for k, v in cfg.items():
    print(f"  {k}: {v}")

## Configuration

- Export: **`messages_2026-10-31`** (change `EXPORT` in Cell 6 for later exports)
- Input: `ankit-iiitdmj/mbg-chat-candidates` → `<EXPORT>/sft_items.jsonl` — only items of kind `teacher`
  (kind `handover` already has its target)
- Teacher: **Qwen2.5-14B-Instruct-AWQ** on the L4 via vLLM, greedy decoding, max 256 new tokens
- Prompt: the student's exact production prompt (`prompts/system_prompt.md`) + `TEACHER_GUIDE` (`src/sft/build.py`)
- Output: `ankit-iiitdmj/mbg-answers-<EXPORT>` → `shards/shard_XXXXXX.jsonl`, then `answers_all.jsonl` + `summary.json`

In [ ]:
# Cell 6 — load training items from the private HF dataset
import json
from collections import Counter
from huggingface_hub import hf_hub_download

EXPORT = "messages_2026-10-31"
_p = hf_hub_download(f"{HF_NS}/mbg-chat-candidates", f"{EXPORT}/sft_items.jsonl", repo_type="dataset")
with open(_p, encoding="utf-8") as fh:
    items = [json.loads(l) for l in fh if l.strip()]
todo = [it for it in items if it["kind"] == "teacher"]

print(f"items: {len(items):,}  ·  teacher items: {len(todo):,}")
print("by kind:", dict(Counter(it["kind"] for it in items)))
print("by language:", dict(Counter(it["lang"] for it in todo)))

In [ ]:
# Cell 7 — resume state from HF Hub (shards already written in earlier sessions)
from judge.hub import JudgeShardManager

ckpt_mgr = JudgeShardManager(HF_NS, EXPORT, prefix="mbg-answers")
done_rows, next_shard = ckpt_mgr.try_load_done()
done_ids = {r["item_id"] for r in done_rows}
remaining = [it for it in todo if it["item_id"] not in done_ids]

print(f"repo: {ckpt_mgr.repo_id} (private)")
print(f"already written: {len(done_rows):,} in {next_shard} shards")
print(f"remaining:       {len(remaining):,}")
if not remaining:
    print("Nothing left — go to Cell 11.")

In [ ]:
# Cell 8 — build the teacher engine (vLLM, else transformers 4-bit)
from judge.engine import build_engine
engine = build_engine(cfg, backend="auto")

In [ ]:
# Cell 9 — answer loop: one shard at a time, pushed to HF before the next; first-shard health check
import time, datetime
from sft.build import teacher_messages, HANDOVER_TAG

SHARD = cfg["shard_size"]
MAX_SHARDS_THIS_SESSION = None   # e.g. 2 to stop early on purpose; None = run until done / disconnected

def health_check(rows):
    n = len(rows)
    empty = sum(1 for r in rows if not (r["answer"] or "").strip()) / n
    ho = sum(1 for r in rows if HANDOVER_TAG in (r["answer"] or "")) / n
    avg = sum(len(r["answer"] or "") for r in rows) / n
    print(f"health: empty={empty:.1%}  handover={ho:.0%}  avg_chars={avg:.0f}  (n={n})")
    if empty > 0.05:
        raise SystemExit(f"ABORT: {empty:.0%} empty replies — check the model/prompt.")
    if not (20 <= avg <= 700):
        raise SystemExit(f"ABORT: average reply length {avg:.0f} chars — check the prompt.")
    if not (0.02 <= ho <= 0.70):
        raise SystemExit(f"ABORT: hand-over share {ho:.0%} — teacher hands over almost always / never.")
    print("health check passed")

t_start = time.time(); written_now = 0; shard_idx = next_shard
for s, i in enumerate(range(0, len(remaining), SHARD)):
    if MAX_SHARDS_THIS_SESSION is not None and s >= MAX_SHARDS_THIS_SESSION:
        print("Reached MAX_SHARDS_THIS_SESSION — stopping cleanly."); break
    batch = remaining[i:i + SHARD]
    t0 = time.time()
    convs = [teacher_messages(it, engine.count_tokens, engine.max_input_tokens) for it in batch]
    outputs = engine.generate_safe(convs)   # never crashes the shard
    rows = [{"item_id": it["item_id"], "answer": out, "engine": engine.name, "model": engine.model_id,
             "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat()}
            for it, out in zip(batch, outputs)]
    dest = ckpt_mgr.save_shard(shard_idx, rows)
    if s == 0:
        health_check(rows)
    written_now += len(rows); shard_idx += 1
    rate = written_now / max(time.time() - t_start, 1e-6)
    left = len(remaining) - (i + len(batch))
    ho = sum(1 for r in rows if HANDOVER_TAG in (r["answer"] or "")) / len(rows)
    print(f"[shard {shard_idx-1}] {len(rows)} replies in {time.time()-t0:.0f}s | handover {ho:.0%} | "
          f"{rate:.1f} items/s | left {left:,} (~{left/max(rate,1e-6)/60:.0f} min) | pushed {dest}")

print(f"\nThis session wrote {written_now:,} replies.")

## Summary + verdict

Re-loads **all** shards from the Hub, writes `answers_all.jsonl` and `summary.json` to
`ankit-iiitdmj/mbg-answers-<EXPORT>`, prints 6 example replies, and gives the verdict.

In [ ]:
# Cell 11 — aggregate all shards, push answers_all.jsonl + summary.json, show examples, verdict
import json, random, tempfile, os

all_rows, _ = ckpt_mgr.try_load_done()
latest = {r["item_id"]: r for r in all_rows}
rows = list(latest.values())
by_id = {it["item_id"]: it for it in todo}

summary = {
    "export": EXPORT, "teacher_items": len(todo), "written": len(rows),
    "empty": sum(1 for r in rows if not (r["answer"] or "").strip()),
    "handover": sum(1 for r in rows if HANDOVER_TAG in (r["answer"] or "")),
    "avg_chars": round(sum(len(r["answer"] or "") for r in rows) / max(len(rows), 1)),
    "teacher": rows[0]["model"] if rows else None,
}
tmp = tempfile.mkdtemp()
with open(os.path.join(tmp, "answers_all.jsonl"), "w", encoding="utf-8") as fh:
    for r in rows:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
with open(os.path.join(tmp, "summary.json"), "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, ensure_ascii=False)
ckpt_mgr.save_file(os.path.join(tmp, "answers_all.jsonl"), "answers_all.jsonl", "all teacher replies")
ckpt_mgr.save_file(os.path.join(tmp, "summary.json"), "summary.json", "answers summary")
print(json.dumps(summary, indent=2, ensure_ascii=False))

random.seed(0)
for r in random.sample(rows, min(6, len(rows))):
    it = by_id[r["item_id"]]
    print(f"\n[{it['lang']} · {it['topic']}] CUSTOMER: {it['messages'][-1]['content'][:200]!r}")
    print(f"  REPLY: {r['answer'][:400]!r}")

done = len(rows) >= len(todo)
print("\nProgress:", f"{len(rows):,}/{len(todo):,} written", "(complete)" if done else "(run Cell 9 again)")
print("VERDICT:", "READY for step 8 (fact check + split, done locally by Claude)" if done else "NOT DONE YET — run Cell 9 again")

## Decision tree

- **READY for step 8** — send the Cell 11 output to Claude. Step 8 runs locally: it fact-checks every reply
  against the memory chunks it was given (numbers, links, times), checks the language, splits by business
  (train 90% / validation 5% / test 5%) and pushes `ankit-iiitdmj/mbg-sft-<EXPORT>`.
- **NOT DONE YET** — the session ended early. Run all again: Cell 7 resumes from the Hub.
- **ABORT in health check** — send the output to Claude; `TEACHER_GUIDE` in `src/sft/build.py` gets fixed.